In [ ]:
%cd ..

In [ ]:
import logging
import os

from openai import AzureOpenAI
from dotenv import load_dotenv

from prompt_forge import (
    Project,
    LLMMessage,
    LLMResponse,
)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)


# ── Azure OpenAI client ───────────────────────────────────────────────────────
class AzureClient:
    ENDPOINT = "https://your-resource.openai.azure.com/"
    API_VERSION = "2024-02-01"
    DEPLOYMENT = "gpt-4o"  # ← change to your deployment name

    def __init__(self):
        self.client = AzureOpenAI(
            api_version=self.API_VERSION,
            azure_endpoint=self.ENDPOINT,
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
        )

    def complete(self, messages: list[LLMMessage], **kwargs) -> LLMResponse:
        # Strip kwargs that AzureOpenAI doesn't accept (e.g. custom keys)
        allowed = {"temperature", "max_tokens", "top_p", "frequency_penalty", "presence_penalty"}
        oai_kwargs = {k: v for k, v in kwargs.items() if k in allowed}

        resp = self.client.chat.completions.create(
            model=self.DEPLOYMENT,
            messages=[{"role": m.role, "content": m.content} for m in messages],
            **oai_kwargs,
        )
        return LLMResponse(
            text=resp.choices[0].message.content,
            usage={
                "input_tokens": resp.usage.prompt_tokens,
                "output_tokens": resp.usage.completion_tokens,
            },
        )


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = "examples/extraction_example"  # Directory containing example subfolders (001/, 002/, ...)
PROJECT_DIR = ".projects/extraction_example"       # Directory to store project files (prompt versions, logs, etc.)

BUNDLE_SCHEMA = {
    "input": ".txt",
    "expected_output": ".json",
}

SEED_PROMPT = (
    "You are a helpful assistant. Process the input and produce the expected output."
)

CONTEXT = (
    "Industrial equipment technical specifications from European manufacturers. Extract structured machine data including power, dimensions, weight, and certifications."
)

# Set to "none" to skip evaluation (faster iteration), or use
# "similarity", "exact_match", "json_fields", "llm_judge"
EVAL_STRATEGY = "json_fields"


In [ ]:
load_dotenv()
llm = AzureClient()
project = Project(
    name="debug_project",
    llm=llm,
    project_dir=PROJECT_DIR,
)

project.set_bundle_schema(**BUNDLE_SCHEMA)
project.set_context(CONTEXT)
project.set_seed_prompt(SEED_PROMPT)

n = project.add_examples_from_directory(DATA_DIR)
print(f"\nLoaded {n} examples")

if n == 0:
    print(
        "\nNo examples found. Create them under:\n"
        f"  {DATA_DIR}/001/input.txt\n"
        f"  {DATA_DIR}/001/expected_output.txt\n"
        "Then re-run."
    )


In [ ]:
from prompt_forge.training.pipeline import TrainingConfig
from prompt_forge.utils import train_val_split

SEED = 42

train_bundles, val_bundles = train_val_split(project.bundles, val_ratio=0.2, seed=SEED)
train_config = TrainingConfig(
    batch_size=20,
    max_iterations=1,
    inference_temperature=1,
    val_max_tokens=5000,
    seed=SEED
)
print(f"\nStarting training on {project.num_examples} examples...")

report = project.train(
    config=train_config,
    eval_strategy=EVAL_STRATEGY,
    train_bundles=train_bundles,
    val_bundles=val_bundles
)

print("\n── Training Report ─────────────────────────────────────────")
print(f"Final version : v{report.final_version}")
print(f"Final score   : {report.final_score}")
print(f"Total tokens  : {report.total_tokens_used:,}")
print(f"Refinement rec: {report.refinement_recommended}")
print()


In [ ]:
for r in report:
    status = "✓" if r.improved else "✗"
    score_str = (
        f"{r.score_before:.3f} → {r.score_after:.3f}"
        if r.score_before is not None and r.score_after is not None
        else "no eval"
    )
    tokens_str = f"  [{r.tokens_used:,} tok]" if r.tokens_used else ""
    print(f"  Iter {r.iteration}: {score_str} {status}{tokens_str}")
    print(f"    Learned: {r.learnings[:120]}")

print("\n── Prompt versions ─────────────────────────────────────────")
for v in project.list_versions():
    score = f"score={v.eval_score:.3f}" if v.eval_score is not None else "no score"
    print(f"  v{v.version}  {score}  {(v.training_log_entry or '')[:80]}")

print("\n── Inference test ──────────────────────────────────────────")
agent = project.get_inference_agent()
print(agent.prompt_info)
result = agent.run(input_text="Hello, this is a test input.")
print(f"Output: {result}")


In [ ]:
print(project.list_versions()[-1].prompt_text)

In [ ]:
print(report.iterations[0].learnings)

In [ ]:
print(report.iterations[0].issues)